# Semantic evidence pilot — nhánh A: Đối chứng huấn luyện tiếp

**Notebook này chạy MỘT nhánh trên HAI GPU bằng DDP.** Chọn **GPU T4 ×2**, bật Internet rồi **Save & Run All**.
Mã nguồn mới đã nhúng sẵn bên trong; không cần tải các tệp `.py` khác lên Kaggle.

10.000 ảnh train + 1.000 ảnh val, cùng cohort ở cả hai nhánh; không dùng test.
1.250 cập nhật, batch toàn cục 8 (4/GPU); 200 cập nhật đầu CE, sau đó CE + SCST + thưởng đối tượng.
Nhánh E thêm cross-attention text→patch và 0.1 × loss cặp giả nhãn (margin 0.2), từ cập nhật đầu. A không có loss này. Chưa chứng minh cải thiện caption hoặc tính mới.

In [ ]:
print('SEMANTIC EVIDENCE A: BOOTSTRAP START', flush=True)
from pathlib import Path

ARM = 'A'
CHECKPOINT = Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT = Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
UPT_PROMPT = Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT = Path('/kaggle/working/semantic_evidence_pilot_A')

# Bạn của bạn sửa các mount ở trên nếu khác; giữ nguyên nội dung đầu vào.
# Không cần COCO_IMAGES hay raw UPT cache: dùng visual HDF5 + prompt JSON đã có.


## 1. Chuẩn bị môi trường và mã nguồn
Kho mã đặt trong `/tmp` để không chiếm danh sách Output. Không cài lại PyTorch của Kaggle.

In [ ]:
import json, os, shutil, subprocess, sys
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', CUBLAS_WORKSPACE_CONFIG=':4096:8',
                  HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='120', TOKENIZERS_PARALLELISM='false',
                  PYTHONHASHSEED='42', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
REPO = Path('/tmp/Image_Captioning-semantic-evidence-pilot')
CHAIR = Path('/tmp/semantic-evidence-pilot-chair')
SETUP_LOG = Path('/tmp/semantic_evidence_pilot_setup_' + ARM + '.log')
PROJECT_COMMIT = '1dd4dc15f42eded978945a3da66c65ec6a725b59'

def run(args, cwd=None, log=None):
    args = list(map(str,args))
    print('\nRunning:', ' '.join(args), flush=True)
    with Path(log or SETUP_LOG).open('a',encoding='utf-8') as output:
        output.write('\nRunning: ' + ' '.join(args) + '\n'); output.flush()
        with subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); output.write(line); output.flush()
            status = process.wait()
    if status: raise RuntimeError(f'Command failed ({status}); full log: {log or SETUP_LOG}')

def checkout(url,path,revision):
    if not path.exists(): run(['git','clone','--no-checkout',url,path])
    assert (path/'.git').is_dir(),path
    run(['git','fetch','--depth','1','origin',revision],cwd=path)
    run(['git','checkout','--detach',revision],cwd=path)

for file in (CHECKPOINT,LEGACY_PROMPT,UPT_PROMPT,COCO_JSON,DETECTIONS):
    assert file.is_file(),f'Missing exact input: {file}. Copy mounted path from Input.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),f'Choose a fresh OUTPUT or restart session: {OUTPUT}'
import torch
assert torch.cuda.device_count()==2,'Select GPU T4 x2: ONE notebook, ONE arm, TWO GPUs.'
for i in range(2): print('GPU',i,torch.cuda.get_device_name(i),flush=True)
checkout('https://github.com/Supzxjee/Image_Captioning.git',REPO,PROJECT_COMMIT)
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
     'h5py','pandas','Pillow','tqdm','scipy','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
     'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])

SOURCES = {'pilot_data.py': '"""Deterministic image cohorts and box pooling; no ML/runtime side effects."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nSEED = 42\nSLOTS = 9\nINPUT_DIM = 768 * 3 + 8\nTRAIN_COUNT = 10000\nVAL_COUNT = 1000\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n\n\ndef cohorts(dataset, train_count=TRAIN_COUNT, val_count=VAL_COUNT):\n    result = {}\n    for split, count in [(\'train\', train_count), (\'val\', val_count)]:\n        rows = []\n        for i, row in enumerate(dataset[\'images\']):\n            selected = row[\'split\'] in (\'train\', \'restval\') if split == \'train\' else row[\'split\'] == \'val\'\n            if selected:\n                rows.append({\'image_id\': i, \'coco_id\': int(Path(row[\'filename\']).stem.rsplit(\'_\', 1)[1]),\n                             \'filename\': row[\'filename\'], \'split\': split})\n        if not 1 < count <= len(rows):\n            raise ValueError(f\'Insufficient {split} images.\')\n        rows.sort(key=lambda r: hashlib.sha256((\'upt-alignment-pilot-v1:\' + split + \':\' + r[\'filename\']).encode()).hexdigest())\n        result[split] = sorted(rows[:count], key=lambda r: r[\'image_id\'])\n    all_rows = result[\'train\'] + result[\'val\']\n    if len({r[\'coco_id\'] for r in all_rows}) != len(all_rows):\n        raise ValueError(\'Overlapping/duplicate COCO IDs.\')\n    return result\n\n\ndef box(value):\n    b = np.asarray(value, dtype=np.float32)\n    if b.shape != (4,) or not np.isfinite(b).all() or (b < 0).any() or (b > 1).any() or (b[2:] <= b[:2]).any():\n        raise ValueError(\'Expected a positive-area normalized xyxy box.\')\n    return b\n\n\ndef pool(patches, bounds):\n    if patches.shape != (196, 768):\n        raise ValueError(\'Expected 14x14 spatial CLIP patch tokens, excluding CLS.\')\n    b = box(bounds)\n    y, x = np.meshgrid((np.arange(14) + .5) / 14, (np.arange(14) + .5) / 14, indexing=\'ij\')\n    x, y = x.ravel(), y.ravel()\n    mask = (x >= b[0]) & (x <= b[2]) & (y >= b[1]) & (y <= b[3])\n    if mask.any():\n        return patches[mask].mean(0)\n    nearest = np.argmin((x - (b[0] + b[2]) / 2)**2 + (y - (b[1] + b[3]) / 2)**2)\n    return patches[nearest].copy()\n\n\ndef geometry(s, o):\n    sw, sh = np.maximum(s[2:] - s[:2], 1e-6)\n    ow, oh = np.maximum(o[2:] - o[:2], 1e-6)\n    sc, oc = (s[:2] + s[2:]) / 2, (o[:2] + o[2:]) / 2\n    inter = np.maximum(np.minimum(s[2:], o[2:]) - np.maximum(s[:2], o[:2]), 0).prod()\n    return np.asarray([(oc[0]-sc[0])/sw, (oc[1]-sc[1])/sh, np.log(ow/sw), np.log(oh/sh),\n                       inter/max(sw*sh+ow*oh-inter, 1e-6), inter/(sw*sh), inter/(ow*oh),\n                       np.linalg.norm(oc-sc)], dtype=np.float32)\n\n\ndef region_slots(entry, visual):\n    """Keep every selected triplet, with separate S/O/R slots; object fallback otherwise."""\n    if visual.shape != (197, 768) or not np.isfinite(visual).all():\n        raise ValueError(\'Invalid visual cache entry.\')\n    features = np.zeros((SLOTS, INPUT_DIM), dtype=np.float32)\n    roles = np.ones(SLOTS, dtype=np.int64)\n    valid = np.zeros(SLOTS, dtype=np.bool_)\n    texts = [\'\'] * SLOTS\n    objects = {o[\'index\']: o for o in entry[\'objects\']}\n    relations = entry[\'relations\']\n    if len(relations) > 3:\n        raise ValueError(\'Expected at most 3 selected relations.\')\n    if relations:\n        for i, relation in enumerate(relations):\n            if relation[\'score\'] < relation[\'threshold\']:\n                raise ValueError(\'A selected relation is below its threshold.\')\n            subject, obj = (objects[relation[key + \'_index\']] for key in (\'subject\', \'object\'))\n            if subject[\'instance_id\'] != relation[\'subject_instance\'] or obj[\'instance_id\'] != relation[\'object_instance\']:\n                raise ValueError(\'Instance-link mismatch.\')\n            s, o = box(subject[\'bbox\']), box(obj[\'bbox\'])\n            union = np.r_[np.minimum(s[:2], o[:2]), np.maximum(s[2:], o[2:])]\n            vs, vo, vu = [pool(visual[1:], b) for b in (s, o, union)]\n            n = i * 3\n            features[n, :768], features[n+1, :768] = vs, vo\n            features[n+2] = np.r_[vs, vo, vu, geometry(s, o)]\n            roles[n:n+3], valid[n:n+3] = (0, 1, 2), True\n            texts[n:n+3] = (subject[\'label\'], obj[\'label\'], relation[\'text\'])\n    else:\n        accepted = sorted((o for o in objects.values() if o[\'confidence\'] >= .5),\n                          key=lambda o: (-o[\'confidence\'], o[\'index\']))[:8]\n        for i, obj in enumerate(accepted):\n            features[i, :768] = pool(visual[1:], obj[\'bbox\'])\n            valid[i], texts[i] = True, obj[\'label\']\n    if bool(valid.any()) != (entry[\'mode\'] != \'no_objects\'):\n        raise ValueError(\'Prompt mode and region slots disagree.\')\n    return features, roles, valid, texts\n\n\ndef evidence_bounds(entry):\n    """Normalized subject/object/allowed boxes, in exactly region_slots order."""\n    result = np.zeros((SLOTS, 3, 4), dtype=np.float32)\n    objects = {o[\'index\']:o for o in entry[\'objects\']}\n    if entry[\'relations\']:\n        for i,r in enumerate(entry[\'relations\']):\n            s,o = (box(objects[r[k+\'_index\']][\'bbox\']) for k in (\'subject\',\'object\'))\n            union=np.r_[np.minimum(s[:2],o[:2]),np.maximum(s[2:],o[2:])]\n            result[i*3]=np.stack((s,o,s));result[i*3+1]=np.stack((s,o,o))\n            result[i*3+2]=np.stack((s,o,union))\n    else:\n        accepted=sorted((o for o in objects.values() if o[\'confidence\']>=.5),key=lambda o:(-o[\'confidence\'],o[\'index\']))[:8]\n        for i,o in enumerate(accepted): result[i]=np.tile(box(o[\'bbox\']),(3,1))\n    return result\n\n\ndef phrase_identity(text):\n    return int(hashlib.sha256(text.strip().casefold().encode()).hexdigest()[:15],16) if text else 0\n', 'pilot_model.py': '"""Text queries image values; no text residual in visual evidence or matching score."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass EvidenceMemory(nn.Module):\n    def __init__(self, dim=512):\n        super().__init__()\n        self.query = nn.Linear(512, dim)\n        self.role = nn.Embedding(3, dim)\n        self.key = nn.Linear(768, dim)\n        self.value = nn.Linear(768, dim)\n        self.position = nn.Linear(4, dim, bias=False)\n        self.norm = nn.LayerNorm(dim)\n        axis = (torch.arange(14, dtype=torch.float32) + .5) / 14\n        y, x = torch.meshgrid(axis, axis, indexing=\'ij\')\n        self.register_buffer(\'xy\', torch.stack((x.flatten(), y.flatten()), -1))\n\n    def attend(self, patches, text, roles, bounds):\n        # bounds: subject, object, allowed box; values contain image features only.\n        xy = self.xy[None].expand(len(text), -1, -1)\n        inside = (xy[:, None] >= bounds[:, :, None, :2]).all(-1) & (xy[:, None] <= bounds[:, :, None, 2:]).all(-1)\n        allowed = inside[:, 2].clone()\n        empty = ~allowed.any(-1)\n        center = (bounds[:, 2, :2] + bounds[:, 2, 2:]) / 2\n        nearest = ((xy-center[:, None])**2).sum(-1).argmin(-1)\n        allowed[torch.arange(len(text), device=text.device)[empty], nearest[empty]] = True\n        spatial = torch.cat((xy, inside[:, :2].transpose(1, 2).float()), -1)\n        q = self.query(text) + self.role(roles)\n        keys = self.key(patches) + self.position(spatial)\n        weights = ((keys*q[:, None]).sum(-1) / q.shape[-1]**.5).masked_fill(~allowed, -torch.inf).softmax(-1)\n        evidence = self.norm((weights[:, :, None]*self.value(patches)).sum(1))\n        score = (F.normalize(q, dim=-1)*F.normalize(evidence, dim=-1)).sum(-1)\n        return evidence, score, weights\n\n    def forward(self, patches, text, roles, valid, bounds):\n        shape = valid.shape\n        tokens, scores, attention = self.attend(patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1))\n        return tokens.reshape(*shape, -1).masked_fill(~valid[..., None], 0), scores.reshape(shape), attention.reshape(*shape,196)\n\n    def alignment(self, patches, text, roles, valid, bounds, phrase_ids, image_ids, positive_scores):\n        """Local-rank pseudo-pair hinge loss; uncertain negatives are a stated limit."""\n        n, slots = valid.shape\n        words, role, ok = phrase_ids.flatten(), roles.flatten(), valid.flatten()\n        owner = torch.arange(n, device=text.device).repeat_interleave(slots)\n        # Exclude every exact phrase known anywhere in the target image, not just its positive slot.\n        known = ((words[None, :, None] == phrase_ids[:, None, :]) & valid[:, None, :]).any(-1)\n        eligible = ok[:, None] & ok[None, :] & (role[:, None] == role[None, :])\n        eligible &= image_ids[owner, None] != image_ids[owner][None, :]\n        eligible &= ~known[owner] & ~known[owner].T\n        has = eligible.any(-1)\n        donor = eligible.long().argmax(-1)\n        index = torch.nonzero(has, as_tuple=False).flatten()\n        if not len(index):\n            return positive_scores.sum()*0, dict(pairs=0, accuracy=None, margin=None)\n        donor = donor[index]\n        p, t, r, b = patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1)\n        _, wrong_text, _ = self.attend(p[index], t[donor], r[index], b[index])\n        _, wrong_region, _ = self.attend(p[donor], t[index], r[index], b[donor])\n        positive = positive_scores.flatten()[index]\n        losses = (F.relu(.2-positive+wrong_text)+F.relu(.2-positive+wrong_region)) / 2\n        margin = torch.minimum(positive-wrong_text, positive-wrong_region)\n        return losses.mean(), dict(pairs=len(index), accuracy=float((margin>0).float().mean().detach()),\n                                   margin=float(margin.mean().detach()))\n\n\nclass PilotModel(nn.Module):\n    def __init__(self, base, arm):\n        super().__init__()\n        if arm not in (\'A\',\'E\'): raise ValueError(arm)\n        self.base, self.arm = base, arm\n        self.extra = EvidenceMemory() if arm==\'E\' else None\n        self.last_gates = None\n        self.last_alignment = dict(loss=0., pairs=0, accuracy=None, margin=None)\n\n    @property\n    def decoder(self): return self.base.decoder\n\n    def memory(self, batch, intervention=\'on\'):\n        memory, padding = self.base.build_memory(batch[\'visual\'], batch[\'legacy\'], batch[\'legacy_mask\'])\n        if self.extra is None: return memory, padding\n        shape = batch[\'valid\'].shape\n        patches = batch[\'visual\'][:, None, 1:].expand(-1, shape[1], -1, -1)\n        text, bounds = batch[\'text\'], batch[\'bounds\']\n        if intervention==\'text_swap\': text=batch[\'swap_text\']\n        elif intervention==\'region_swap\':\n            joint = batch[\'swap_valid\']\n            patches = torch.where(joint[...,None,None], batch[\'swap_visual\'][:,None,1:].expand_as(patches), patches)\n            bounds = torch.where(joint[...,None,None], batch[\'swap_bounds\'], bounds)\n        elif intervention not in (\'on\',\'off\'): raise ValueError(intervention)\n        tokens, scores, attention = self.extra(patches, text, batch[\'roles\'], batch[\'valid\'], bounds)\n        self.last_gates = torch.ones_like(scores)\n        self.last_attention = attention.detach()\n        self._evidence_inputs = (patches, text, batch[\'roles\'], batch[\'valid\'], bounds, scores)\n        extra_padding = ~batch[\'valid\']\n        if intervention==\'off\':\n            tokens=torch.zeros_like(tokens);extra_padding=torch.ones_like(extra_padding)\n        return torch.cat((memory,tokens),1), torch.cat((padding,extra_padding),1)\n\n    def teacher(self, memory, padding, targets, masks=None):\n        b, slots, dim = memory.shape\n        repeats = targets.size(0)//b\n        if repeats*b != targets.size(0): raise ValueError(\'Caption/image batch mismatch\')\n        m=memory[:,None].expand(b,repeats,slots,dim).reshape(-1,slots,dim)\n        p=padding[:,None].expand(b,repeats,slots).reshape(-1,slots)\n        return self.decoder(targets,m,tgt_key_padding_mask=masks,memory_key_padding_mask=p)\n\n    def forward(self, batch, sample, active, advantage, scst_weight, pad_idx):\n        from captioning.scst import policy_loss\n        memory,padding=self.memory(batch)\n        targets=batch[\'captions\'].flatten(0,1); masks=batch[\'caption_mask\'].flatten(0,1)\n        logits=self.teacher(memory,padding,targets[:,:-1],masks[:,:-1])\n        ce=F.cross_entropy(logits.flatten(0,1),targets[:,1:].reshape(-1),ignore_index=pad_idx)\n        rl=ce.new_zeros(())\n        if scst_weight:\n            rl=policy_loss(self.teacher(memory,padding,sample[:,:-1]),sample[:,1:],active,advantage)\n        alignment=ce.new_zeros(())\n        info=dict(pairs=0,accuracy=None,margin=None)\n        if self.extra is not None:\n            p,t,r,v,b,s=self._evidence_inputs\n            alignment,info=self.extra.alignment(p,t,r,v,b,batch[\'phrase_ids\'],batch[\'image_id\'],s)\n        self.last_alignment=dict(loss=float(alignment.detach()),**info)\n        return ce+scst_weight*rl+.1*alignment, ce.detach(), rl.detach()\n', 'run_pilot.py': '"""Manual, one-arm, two-GPU UPT caption pilot. Importing this file never trains."""\nimport argparse\nfrom collections import Counter\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nimport numpy as np\n\nfrom pilot_data import cohorts, digest, identity, region_slots, evidence_bounds, phrase_identity, save, SLOTS, INPUT_DIM\n\nCLIP = \'openai/clip-vit-base-patch16\'\nCLIP_REVISION = \'57c216476eefef5ab752ec549e440a49ae4ae5f3\'\nCHECKPOINT_SHA = \'5f8eead3f4e960189345b1a9bf492763fe52ec71f70115b73244ffe9d03d2e75\'\n\n\ndef prepare(config_path):\n    import h5py\n    import torch\n    from transformers import CLIPTextModel, AutoTokenizer\n    from captioning.visual_cache import VisualCache\n    from captioning.semantic_prompts import align_prompt_cache\n    from captioning.object_reward import make_targets, _AnnotationReader\n    from captioning.scst_runtime import configure_strict_runtime\n    cfg = json.loads(Path(config_path).read_text())\n    out = Path(cfg[\'output\'])\n    if out.exists():\n        raise FileExistsError(f\'Use a new OUTPUT: {out}\')\n    for key in (\'checkpoint\', \'legacy_prompt\', \'upt_prompt\', \'dataset_json\', \'detections\'):\n        if not Path(cfg[key]).is_file():\n            raise FileNotFoundError(f\'{key}: {cfg[key]}\')\n    hashes = {}\n    for key in (\'checkpoint\', \'legacy_prompt\', \'upt_prompt\', \'dataset_json\', \'detections\'):\n        print(f\'Checking pinned input: {key}\', flush=True)\n        hashes[key] = digest(cfg[key])\n    if hashes != cfg[\'expected_hashes\']:\n        raise ValueError(f\'Pinned inputs differ: {hashes}. Do not silently replace the baseline/cache.\')\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    groups = cohorts(dataset)\n    all_rows = groups[\'train\'] + groups[\'val\']\n    prompts = json.loads(Path(cfg[\'upt_prompt\']).read_text())\n    if len(prompts) != 123287 or any(r[\'filename\'] not in prompts for r in all_rows):\n        raise ValueError(\'Incomplete UPT prompt mapping.\')\n    for rows in groups.values():\n        for row in rows:\n            row[\'mode\'] = prompts[row[\'filename\']][\'mode\']\n    if identity(groups) != \'6bbdf3b75869927334ec6c2f868edf9f52346c0e79401aa0e118ba5a02fb711b\':\n        raise ValueError(\'Cohort differs from the locally reviewed 10000/1000 image manifest.\')\n    chair = Path(cfg[\'chair\'])\n    if digest(chair / \'chair.pkl\') != \'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\':\n        raise ValueError(\'Pinned CHAIR annotation cache differs.\')\n    # Read validation annotation coverage for evaluation only, not training targets.\n    with (chair / \'chair.pkl\').open(\'rb\') as stream:\n        annotation = _AnnotationReader(stream).load()\n    if any(r[\'coco_id\'] not in annotation.imid_to_objects for r in groups[\'val\']):\n        raise ValueError(\'CHAIR validation coverage incomplete.\')\n    del annotation\n    out.mkdir(parents=True)\n    cfg.update(groups=groups, cohort_sha256=identity(groups), test_used=False,\n               alignment_weight=.1 if cfg[\'arm\']==\'E\' else 0., alignment_margin=.2,\n               negative_policy=\'Within-rank same-role, other image, exclude exact known phrases in either image; deterministic first eligible; may contain false negatives\',\n               validation_scope=\'Historically reused development validation, not an independent unseen test\',\n               source_checkpoint_sha256=hashes[\'checkpoint\'], supported_actions=116,\n               loss=\'CE + scheduled 0.05*SCST(CIDEr-D+0.5*object_count_v2) + E-only 0.1*pseudo_pair_hinge_margin0.2\',\n               evaluation_policy=\'Raw greedy, all arms and start checkpoint; no reranking in this screening pilot\',\n               comparison_limits=\'A continued-training control; E text-query patch attention plus alignment loss. Whole-method comparison only; UPT pseudo-pairs and cross-image negatives can be noisy.\',\n               module_claim=\'Text-conditioned visual evidence with pseudo-pair supervision; not calibrated relation truth or established novelty\')\n    save(out / \'groups.json\', groups)\n    targets = make_targets(groups, dataset, chair / \'chair.pkl\', json.loads(Path(cfg[\'detections\']).read_text()),\n                           \'aliases_v2\', \'unique_count_v2\')\n    save(out / \'train_object_targets.json\', targets)\n    cfg[\'object_targets_sha256\'] = digest(out / \'train_object_targets.json\')\n    bundle = torch.load(cfg[\'legacy_prompt\'], map_location=\'cpu\', weights_only=False)\n    names = [str(Path(\'/images\') / dataset[\'images\'][r[\'image_id\']][\'filepath\'] / r[\'filename\']) for r in all_rows]\n    legacy = align_prompt_cache(bundle[\'data\'], names)\n    if len(legacy) != len(all_rows):\n        raise ValueError(\'Missing baseline prompt entries; UPT JSON must not replace this cache.\')\n    first = legacy[names[0]]\n    length = first[\'tokens\'].shape[0]\n    if first[\'tokens\'].shape != (length, 512):\n        raise ValueError(\'Legacy prompt shape differs.\')\n    stage = Path(cfg[\'stage\'])\n    if stage.exists():\n        raise FileExistsError(f\'Use a fresh stage path: {stage}\')\n    stage.parent.mkdir(parents=True, exist_ok=True)\n    required = len(all_rows) * (197 * 768 + SLOTS * INPUT_DIM + length * 512 + SLOTS * 512) * 4\n    if shutil.disk_usage(stage.parent).free < required + 2 * 1024**3:\n        raise RuntimeError(\'Not enough disk space to stage the 11000-image subset without quantizing.\')\n    cache = VisualCache([cfg[\'visual_cache\']], id_key=\'coco_id\')\n    cache.require_ids([r[\'coco_id\'] for r in all_rows], \'pilot train/val\')\n    text_rows = []\n    n = len(all_rows)\n    try:\n        with h5py.File(stage, \'w\') as f:\n            arrays = {\'visual\': ((n, 197, 768), \'f4\'), \'legacy\': ((n, length, 512), \'f4\'),\n                      \'legacy_mask\': ((n, length), \'i8\'), \'regions\': ((n, SLOTS, INPUT_DIM), \'f4\'),\n                      \'roles\': ((n, SLOTS), \'i8\'), \'valid\': ((n, SLOTS), \'?\'),\n                      \'text\': ((n, SLOTS, 512), \'f4\'), \'bounds\': ((n,SLOTS,3,4),\'f4\'), \'phrase_ids\': ((n,SLOTS),\'i8\')}\n            for key, (shape, dtype) in arrays.items():\n                f.create_dataset(key, shape=shape, dtype=dtype, chunks=(1, *shape[1:]))\n            # Read source HDF5 in source-row order rather than random image order.\n            order = sorted(range(n), key=lambda i: cache.index[all_rows[i][\'coco_id\']])\n            text_rows = [None] * n\n            for count, i in enumerate(order, 1):\n                row = all_rows[i]\n                visual = cache.read(row[\'coco_id\'])\n                features, roles, valid, texts = region_slots(prompts[row[\'filename\']], visual)\n                old = legacy[names[i]]\n                if old[\'tokens\'].shape != (length, 512) or old[\'mask\'].shape != (length,):\n                    raise ValueError(\'Inconsistent legacy prompt length.\')\n                for key, value in [(\'visual\', visual), (\'legacy\', old[\'tokens\'].float().numpy()),\n                                   (\'legacy_mask\', old[\'mask\'].long().numpy()), (\'regions\', features),\n                                   (\'roles\', roles), (\'valid\', valid), (\'bounds\',evidence_bounds(prompts[row[\'filename\']])),\n                                   (\'phrase_ids\',np.asarray([phrase_identity(t) for t in texts],dtype=np.int64))]:\n                    f[key][i] = value\n                text_rows[i] = texts\n                if count % 250 == 0 or count == n:\n                    print(f\'Staged spatial tokens/regions: {count}/{n}\', flush=True)\n    finally:\n        cache.close()\n    del bundle, legacy, prompts, targets\n    # Frozen EOS hidden states (512D), shared across repeated phrases; no trainable text encoder.\n    torch.manual_seed(42)\n    configure_strict_runtime(True)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError(\'Select GPU T4 x2. This notebook runs ONE arm using two GPUs.\')\n    encoder = CLIPTextModel.from_pretrained(CLIP, revision=CLIP_REVISION).cuda().eval()\n    tokenizer = AutoTokenizer.from_pretrained(CLIP, revision=CLIP_REVISION)\n    unique = sorted({t for row in text_rows for t in row if t})\n    embedding = {}\n    with torch.no_grad():\n        for start in range(0, len(unique), 128):\n            batch = unique[start:start+128]\n            tokens = tokenizer(batch, padding=True, truncation=True, max_length=77, return_tensors=\'pt\').to(\'cuda\')\n            vectors = encoder(**tokens).pooler_output.float().cpu().numpy()\n            for text, vector in zip(batch, vectors):\n                embedding[text] = vector\n            if start % 1024 == 0:\n                print(f\'Frozen text embeddings: {min(start+128, len(unique))}/{len(unique)}\', flush=True)\n    with h5py.File(stage, \'r+\') as f:\n        for i, texts in enumerate(text_rows):\n            values = np.zeros((SLOTS, 512), dtype=np.float32)\n            for j, text in enumerate(texts):\n                if text:\n                    values[j] = embedding[text]\n            f[\'text\'][i] = values\n        f.attrs[\'cohort_sha256\'] = cfg[\'cohort_sha256\']\n        f.attrs[\'complete\'] = True\n    cfg[\'stage_sha256\'] = digest(stage)\n    cfg[\'counts_by_mode\'] = {key: dict(Counter(r[\'mode\'] for r in rows)) for key, rows in groups.items()}\n    save(out / \'slot_texts.json\', [{\'filename\': r[\'filename\'], \'texts\': texts} for r, texts in zip(all_rows, text_rows)])\n    save(out / \'protocol.json\', cfg)\n    print(\'PREPARATION PASS:\', json.dumps(cfg[\'counts_by_mode\']), \'cohort:\', cfg[\'cohort_sha256\'], flush=True)\n\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef build_model(cfg, tokenizer):\n    from captioning.models import UniversalVisionEncoder, CaptionDecoder, ImageCaptioningModel\n    from captioning.checkpoints import load_checkpoint\n    from pilot_model import PilotModel\n    base = ImageCaptioningModel(UniversalVisionEncoder(load_backbone=False, visual_adapter=\'qformer\',\n                               num_visual_queries=32, qformer_layers=2), CaptionDecoder(len(tokenizer.word2idx)))\n    load_checkpoint(base, cfg[\'checkpoint\'], tokenizer)\n    return PilotModel(base, cfg[\'arm\']).cuda().eval()\n\n\ndef corpus(cfg):\n    from captioning.tokenizer import CaptionTokenizer\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    train = [r for r in dataset[\'images\'] if r[\'split\'] in (\'train\', \'restval\')]\n    tokenizer = CaptionTokenizer([s[\'raw\'] for r in train for s in r[\'sentences\'][:5]])\n    return dataset, tokenizer\n\n\ndef make_loader(cfg, dataset, tokenizer, split, rank, world):\n    import h5py\n    import torch\n    from torch.utils.data import Dataset, DataLoader, DistributedSampler\n    rows = cfg[\'groups\'][split]\n    train_count = len(cfg[\'groups\'][\'train\'])\n\n    class Staged(Dataset):\n        def __init__(self):\n            self.f = h5py.File(cfg[\'stage\'], \'r\')\n            if not self.f.attrs.get(\'complete\') or self.f.attrs[\'cohort_sha256\'] != cfg[\'cohort_sha256\']:\n                raise ValueError(\'Incomplete/different staged cohort.\')\n\n        def __len__(self):\n            return len(rows)\n\n        def __getitem__(self, index):\n            row = rows[index]\n            i = index if split == \'train\' else index + train_count\n            result = {key: torch.from_numpy(self.f[key][i]) for key in\n                      (\'visual\', \'legacy\', \'legacy_mask\', \'roles\', \'valid\', \'text\', \'bounds\', \'phrase_ids\')}\n            refs = dataset[\'images\'][row[\'image_id\']][\'sentences\'][:5]\n            encoded = [tokenizer.encode(s[\'raw\']) for s in refs]\n            result[\'captions\'] = torch.stack([x[0] for x in encoded])\n            result[\'caption_mask\'] = torch.stack([x[1] for x in encoded])\n            result[\'image_id\'] = torch.tensor(row[\'image_id\'])\n            # Same-mode deterministic donor. Missing slots are not filled with fake evidence.\n            candidates = donors[row[\'mode\']]\n            pos = candidates.index(index)\n            donor = candidates[(pos + 1) % len(candidates)]\n            di = donor if split == \'train\' else donor + train_count\n            donor_valid = self.f[\'valid\'][di]\n            result[\'swap_visual\']=torch.from_numpy(self.f[\'visual\'][di])\n            result[\'swap_bounds\']=torch.from_numpy(self.f[\'bounds\'][di])\n            joint = self.f[\'valid\'][i] & donor_valid\n            result[\'swap_text\'] = torch.from_numpy(np.where(joint[:, None], self.f[\'text\'][di], self.f[\'text\'][i]))\n            result[\'swap_valid\']=torch.from_numpy(joint)\n            return result\n\n    donors = {mode: [i for i, r in enumerate(rows) if r[\'mode\'] == mode] for mode in set(r[\'mode\'] for r in rows)}\n    staged = Staged()\n    if split == \'train\':\n        sampler = DistributedSampler(staged, num_replicas=world, rank=rank, shuffle=True, seed=42, drop_last=False)\n        return DataLoader(staged, batch_size=cfg[\'global_batch_size\']//world, sampler=sampler,\n                          num_workers=0, pin_memory=True, drop_last=False)\n    # Explicit rank slice: no padding or duplicate validation IDs from DistributedSampler.\n    subset = torch.utils.data.Subset(staged, list(range(rank, len(rows), world)))\n    return DataLoader(subset, batch_size=cfg[\'eval_batch_size\'], shuffle=False, num_workers=0, pin_memory=True)\n\n\ndef cuda_batch(batch):\n    return {key: value.cuda(non_blocking=True) for key, value in batch.items()}\n\n\ndef ground_truth(cfg, dataset, out):\n    rows = cfg[\'groups\'][\'val\']\n    gt = {\'info\': {}, \'licenses\': [], \'type\': \'captions\', \'images\': [{\'id\': r[\'image_id\']} for r in rows],\n          \'annotations\': [{\'id\': i*5+j, \'image_id\': r[\'image_id\'], \'caption\': ref[\'raw\']}\n                          for i, r in enumerate(rows) for j, ref in enumerate(dataset[\'images\'][r[\'image_id\']][\'sentences\'][:5])]}\n    save(out / \'val_gt.json\', gt)\n\n\ndef predict(model, loader, tokenizer, intervention):\n    import torch\n    from captioning.scst import rollout, decode\n    records, diagnostics = [], []\n    with torch.no_grad():\n        for index, batch in enumerate(loader):\n            batch = cuda_batch(batch)\n            memory, padding = model.memory(batch, intervention)\n            seq, _ = rollout(model, memory, padding, tokenizer, sample=False)\n            texts = decode(seq, tokenizer)\n            target = batch[\'captions\'].flatten(0, 1)\n            mask = batch[\'caption_mask\'].flatten(0, 1)\n            logits = model.teacher(memory, padding, target[:, :-1], mask[:, :-1])\n            logp = logits.log_softmax(-1).gather(-1, target[:, 1:, None]).squeeze(-1)\n            valid = target[:, 1:].ne(tokenizer.pad_idx)\n            nll = (-(logp * valid).sum(1) / valid.sum(1).clamp_min(1)).reshape(-1, 5).mean(1).cpu().tolist()\n            ids = batch[\'image_id\'].cpu().tolist()\n            for i, (image_id, caption, value) in enumerate(zip(ids, texts, nll)):\n                records.append({\'image_id\': image_id, \'caption\': caption})\n                row = {\'image_id\': image_id, \'reference_nll_per_token\': value}\n                if model.extra is not None:\n                    v = batch[\'valid\'][i]\n                    row[\'valid_evidence_slots\']=int(v.sum())\n                    att=model.last_attention[i]\n                    row[\'attention_entropy\']=float((-(att.clamp_min(1e-12).log()*att).sum(-1))[v].mean()) if v.any() else None\n                diagnostics.append(row)\n            if index % 10 == 0:\n                print(f\'Evaluation {intervention}: {len(records)} local images\', flush=True)\n    return records, diagnostics\n\n\ndef worker(protocol):\n    import torch\n    import torch.distributed as dist\n    from torch.nn.parallel import DistributedDataParallel as DDP\n    from captioning.scst import TrainingCider, rollout, decode\n    from captioning.object_reward import TrainingObjectReward, combine_rewards\n    from captioning.scst_runtime import configure_strict_runtime, runtime_manifest, model_digest\n    cfg = json.loads(Path(protocol).read_text())\n    rank, world = int(os.environ[\'RANK\']), int(os.environ[\'WORLD_SIZE\'])\n    local_rank = int(os.environ[\'LOCAL_RANK\'])\n    if world != 2:\n        raise RuntimeError(\'This pilot requires exactly two GPU ranks for ONE arm.\')\n    torch.cuda.set_device(local_rank)\n    torch.set_num_threads(2)\n    configure_strict_runtime(True)\n    # Both GPUs belong to this single arm; NCCL synchronizes its gradients.\n    from datetime import timedelta\n    dist.init_process_group(\'nccl\', timeout=timedelta(minutes=20))\n    out = Path(cfg[\'output\'])\n    save(out / f\'runtime_rank{rank}.json\', runtime_manifest())\n    seed_all(42)\n    dataset, tokenizer = corpus(cfg)\n    model = build_model(cfg, tokenizer)\n    initial_base_hash = model_digest(model.base)\n    train = make_loader(cfg, dataset, tokenizer, \'train\', rank, world)\n    validation = make_loader(cfg, dataset, tokenizer, \'val\', rank, world)\n    if len(train) != math.ceil(len(cfg[\'groups\'][\'train\'])/cfg[\'global_batch_size\']):\n        raise ValueError(\'Unexpected DDP training step count.\')\n    # Inspect the frozen starting system with extra tokens masked off, on the same GPU/profile.\n    records, detail = predict(model, validation, tokenizer, \'off\')\n    save(out / f\'origin_rank{rank}.json\', records)\n    save(out / f\'origin_diagnostic_rank{rank}.json\', detail)\n    # Every rank uses identical source initialization; different sampling RNG streams are explicit.\n    seed_all(42 + rank)\n    if digest(out / \'train_object_targets.json\') != cfg[\'object_targets_sha256\']:\n        raise ValueError(\'Object reward targets changed.\')\n    targets = json.loads((out / \'train_object_targets.json\').read_text())[\'data\']\n    object_reward = TrainingObjectReward(targets, \'aliases_v2\', \'unique_count_v2\')\n    # Retain full TRAIN IDF like the successful full-train baseline, even in this 10k pilot.\n    references = {i: [s[\'raw\'] for s in row[\'sentences\'][:5]] for i, row in enumerate(dataset[\'images\'])\n                  if row[\'split\'] in (\'train\', \'restval\')}\n    print(f\'[rank {rank}] Building fixed TRAIN CIDEr-D IDF: {len(references)} images; CPU work, please wait.\', flush=True)\n    reward = TrainingCider(references, reference_cache_size=256)\n    print(f\'[rank {rank}] TRAIN reward ready; starting synchronized updates.\', flush=True)\n    params = [{\'params\': [p for p in model.base.parameters() if p.requires_grad], \'lr\': cfg[\'base_lr\']}]\n    if model.extra is not None:\n        params.append({\'params\': list(model.extra.parameters()), \'lr\': cfg[\'branch_lr\']})\n    optimizer = torch.optim.AdamW(params)\n    # No collective calls during sampled/greedy decoding: those use the unwrapped model in no_grad.\n    # All ranks execute exactly ONE synchronized loss backward per update.\n    ddp = DDP(model, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False,\n              find_unused_parameters=True)\n    history = []\n    begin = time.monotonic()\n    for step, batch in enumerate(train, 1):\n        batch = cuda_batch(batch)\n        optimizer.zero_grad(set_to_none=True)\n        weight = 0. if step <= cfg[\'warmup_updates\'] else cfg[\'scst_weight\']\n        sample = active = advantage = None\n        record = {\'step\': step, \'global_images_seen\': min(step*cfg[\'global_batch_size\'], 10000),\n                  \'scst_weight\': weight}\n        if weight:\n            with torch.no_grad():\n                memory, padding = model.memory(batch)\n                sample, active = rollout(model, memory, padding, tokenizer, sample=True)\n                greedy, _ = rollout(model, memory, padding, tokenizer, sample=False)\n            ids = batch[\'image_id\'].cpu().tolist()\n            sampled_text, greedy_text = decode(sample, tokenizer), decode(greedy, tokenizer)\n            sample_cider, greedy_cider = reward.score(ids, sampled_text), reward.score(ids, greedy_text)\n            sample_object, _ = object_reward.score(ids, sampled_text)\n            greedy_object, _ = object_reward.score(ids, greedy_text)\n            sample_reward = combine_rewards(sample_cider, sample_object, .5)\n            greedy_reward = combine_rewards(greedy_cider, greedy_object, .5)\n            advantage = torch.from_numpy(sample_reward - greedy_reward).cuda()\n            record.update(sample_reward=float(sample_reward.mean()), greedy_reward=float(greedy_reward.mean()),\n                          sample_cider=float(sample_cider.mean()), greedy_cider=float(greedy_cider.mean()),\n                          sample_object=float(sample_object.mean()), greedy_object=float(greedy_object.mean()))\n        loss, ce, rl = ddp(batch, sample, active, advantage, weight, tokenizer.pad_idx)\n        if not torch.isfinite(loss):\n            raise FloatingPointError(f\'Nonfinite loss, rank {rank}, step {step}.\')\n        loss.backward()\n        branch_grad = (sum(float(p.grad.detach().square().sum()) for p in model.extra.parameters() if p.grad is not None)**.5\n                       if model.extra is not None else 0.)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n        optimizer.step()\n        record.update(alignment=model.last_alignment,\n                      submodule_grad_norms={name:sum(float(p.grad.detach().square().sum()) for p in part.parameters() if p.grad is not None)**.5 for name,part in model.extra.named_children()} if model.extra is not None else {},\n                      loss=float(loss.detach()), ce_loss=float(ce), scst_loss=float(rl),\n                      branch_grad_norm=branch_grad, grad_norm=float(grad_norm), seconds=time.monotonic()-begin)\n        history.append(record)\n        if step == 1 or step % 25 == 0 or step == len(train):\n            save(out / f\'history_rank{rank}.json\', history)\n            print(f\'[{cfg["arm"]} rank {rank}] {json.dumps(record)}\', flush=True)\n    if rank == 0:\n        state = {\'format\': \'semantic_evidence_pilot_v1\',\n                 \'model_state_dict\': {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                 \'arm\': cfg[\'arm\'], \'caption_word2idx\': tokenizer.word2idx, \'cohort_sha256\': cfg[\'cohort_sha256\'],\n                 \'source_checkpoint_sha256\': cfg[\'source_checkpoint_sha256\'],\n                 \'protocol_sha256\': digest(protocol), \'completed_updates\': len(train)}\n        temporary = out / \'checkpoint.pth.partial\'\n        # Legacy serialization avoids Kaggle treating a .pth checkpoint as a ZIP to unpack.\n        torch.save(state, temporary, _use_new_zipfile_serialization=False)\n        temporary.replace(out / \'checkpoint.pth\')\n    # Evaluation has no DDP communication; different rank runtimes cannot stall loss collectives.\n    conditions = [\'on\'] if cfg[\'arm\'] == \'A\' else [\'on\', \'off\', \'text_swap\', \'region_swap\']\n    for condition in conditions:\n        records, detail = predict(model, validation, tokenizer, condition)\n        save(out / f\'{condition}_rank{rank}.json\', records)\n        save(out / f\'{condition}_diagnostic_rank{rank}.json\', detail)\n    save(out / f\'worker_complete_rank{rank}.json\', {\'status\': \'COMPLETE\', \'updates\': len(train),\n         \'initial_base_sha256\': initial_base_hash, \'rank\': rank, \'source_arm\': cfg[\'arm\'],\n         \'trainable_base_parameters\': sum(p.numel() for p in model.base.parameters() if p.requires_grad),\n         \'trainable_branch_parameters\': sum(p.numel() for p in model.extra.parameters()) if model.extra is not None else 0,\n         \'test_used\': False})\n    dist.destroy_process_group()\n\n\ndef evaluate(protocol):\n    from captioning.metrics import score_files\n    cfg = json.loads(Path(protocol).read_text())\n    out = Path(cfg[\'output\'])\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    ground_truth(cfg, dataset, out)\n    expected = {r[\'image_id\']: r for r in cfg[\'groups\'][\'val\']}\n    conditions = [\'origin\', \'on\'] + ([] if cfg[\'arm\'] == \'A\' else [\'off\', \'text_swap\', \'region_swap\'])\n    results, predictions, details = {}, {}, {}\n    for condition in conditions:\n        rows, detail = [], []\n        for rank in (0, 1):\n            completion = json.loads((out / f\'worker_complete_rank{rank}.json\').read_text())\n            if completion[\'status\'] != \'COMPLETE\':\n                raise RuntimeError(\'A GPU worker did not finish.\')\n            rows.extend(json.loads((out / f\'{condition}_rank{rank}.json\').read_text()))\n            detail.extend(json.loads((out / f\'{condition}_diagnostic_rank{rank}.json\').read_text()))\n        rows.sort(key=lambda r: r[\'image_id\'])\n        if len(rows) != len(expected) or {r[\'image_id\'] for r in rows} != expected.keys():\n            raise ValueError(\'Missing/duplicated validation predictions.\')\n        path = out / f\'{condition}_predictions.json\'\n        save(path, rows)\n        predictions[condition] = {r[\'image_id\']: r[\'caption\'] for r in rows}\n        details[condition] = {r[\'image_id\']: r for r in detail}\n        if len(details[condition]) != len(expected):\n            raise ValueError(\'Incomplete validation diagnostics.\')\n        # Score branch removal as well: a caption change alone is not evidence of a useful branch.\n        if condition not in (\'origin\', \'on\', \'off\'):\n            continue\n        caption = score_files(path, out / \'val_gt.json\', out / f\'{condition}_metrics.json\')\n        chair_input = out / f\'{condition}_chair_input.json\'\n        chair_output = out / f\'{condition}_chair_output.json\'\n        save(chair_input, [{\'image_id\': expected[r[\'image_id\']][\'coco_id\'], \'caption\': r[\'caption\']} for r in rows])\n        subprocess.run([sys.executable, \'-u\', str(Path(cfg[\'chair\'])/\'chair.py\'), \'--cap_file\', str(chair_input),\n                        \'--cache\', str(Path(cfg[\'chair\'])/\'chair.pkl\'), \'--save_path\', str(chair_output)],\n                       cwd=cfg[\'chair\'], check=True)\n        chair = json.loads(chair_output.read_text())\n        if {r[\'image_id\'] for r in chair[\'sentences\']} != {r[\'coco_id\'] for r in expected.values()}:\n            raise ValueError(\'CHAIR IDs differ.\')\n        # Per-image CIDEr uses the SAME full validation IDF for all mode subsets.\n        from pycocotools.coco import COCO\n        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n        from pycocoevalcap.cider.cider import Cider\n        gt = COCO(str(out / \'val_gt.json\'))\n        tokenizer = PTBTokenizer()\n        ids = [r[\'image_id\'] for r in rows]\n        refs = tokenizer.tokenize({i: gt.imgToAnns[i] for i in ids})\n        hyp = tokenizer.tokenize({r[\'image_id\']: [{\'caption\': r[\'caption\']}] for r in rows})\n        _, per_image = Cider().compute_score(refs, hyp)\n        grouped = {}\n        for mode in (\'object_relation\', \'object_only\', \'no_objects\'):\n            indices = [i for i, row in enumerate(rows) if expected[row[\'image_id\']][\'mode\'] == mode]\n            grouped[mode] = {\'images\': len(indices), \'CIDEr_full_val_IDF\': float(np.mean(np.asarray(per_image)[indices])) if indices else None}\n        save(out / f\'{condition}_per_image_cider.json\', [{\'image_id\': r[\'image_id\'], \'CIDEr\': float(v)} for r, v in zip(rows, per_image)])\n        results[condition] = {\'caption\': caption, \'chair\': chair[\'overall_metrics\'], \'by_mode\': grouped}\n    sensitivity = {}\n    for condition in conditions:\n        if condition in (\'origin\', \'on\'):\n            continue\n        sensitivity[condition] = {}\n        for mode in (\'all\', \'object_relation\', \'object_only\', \'no_objects\'):\n            ids = [i for i, row in expected.items() if mode == \'all\' or row[\'mode\'] == mode]\n            sensitivity[condition][mode] = {\'images\': len(ids),\n                \'changed_captions\': sum(predictions[condition][i] != predictions[\'on\'][i] for i in ids),\n                \'mean_reference_nll_delta\': float(np.mean([details[condition][i][\'reference_nll_per_token\'] -\n                                                          details[\'on\'][i][\'reference_nll_per_token\'] for i in ids])) if ids else None}\n    summary = {\'status\': \'PILOT_COMPLETE\', \'arm\': cfg[\'arm\'], \'metrics\': results, \'sensitivity\': sensitivity,\n               \'cohort_sha256\': cfg[\'cohort_sha256\'], \'stage_sha256\': cfg[\'stage_sha256\'],\n               \'protocol_sha256\': digest(protocol), \'checkpoint_sha256\': digest(out / \'checkpoint.pth\'),\n               \'test_used\': False, \'selection\': \'End checkpoint only, fixed before metrics; no automatic promotion\',\n               \'policy\': cfg[\'evaluation_policy\'], \'limits\': [cfg[\'validation_scope\'], cfg[\'comparison_limits\'],\n               \'Greedy pilot metrics must not be compared directly to historical beam5/fixed test metrics.\',\n               \'Text/region swaps are sensitivity probes, not verified false relations.\',\n               \'CHAIR measures objects, not predicate correctness; contrastive scores are not calibrated probabilities.\',\n               \'Different accounts/GPUs may differ numerically; compare input/cohort/runtime manifests.\']}\n    save(out / \'run_summary.json\', summary)\n    print(json.dumps(summary, indent=2), flush=True)\n\n\ndef archive(protocol):\n    cfg = json.loads(Path(protocol).read_text())\n    out = Path(cfg[\'output\'])\n    with zipfile.ZipFile(str(out) + \'_review.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(out.rglob(\'*\')):\n            if path.is_file() and path.suffix in (\'.json\', \'.jsonl\', \'.log\', \'.py\'):\n                z.write(path, path.relative_to(out))\n    print(\'Return:\', str(out) + \'_review.zip\', flush=True)\n    print(\'Keep checkpoint separately in saved notebook output:\', out / \'checkpoint.pth\', flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'mode\', choices=[\'prepare\', \'worker\', \'evaluate\', \'archive\'])\n    parser.add_argument(\'--config\', required=True)\n    args = parser.parse_args()\n    globals()[args.mode](args.config)\n', 'test_pilot_data.py': "import unittest\nimport numpy as np\nfrom pilot_data import box, pool, geometry, region_slots, cohorts, identity, evidence_bounds, phrase_identity\n\n\nclass DataChecks(unittest.TestCase):\n    def test_spatial_pool_and_tiny_box(self):\n        patches = np.repeat(np.arange(196, dtype=np.float32)[:, None], 768, axis=1)\n        np.testing.assert_allclose(pool(patches, [0, 0, 1, 1]), 97.5)\n        np.testing.assert_allclose(pool(patches, [0, 0, .001, .001]), 0)\n        np.testing.assert_allclose(geometry(box([0, 0, 1, 1]), box([0, 0, 1, 1])), [0,0,0,0,1,1,1,0])\n        for b in ([0,0,0,1], [-1,0,1,1], [0,0,float('nan'),1]):\n            with self.assertRaises(ValueError): box(b)\n\n    def test_instance_link_fallback_and_no_objects(self):\n        objects = [dict(index=0, instance_id='p0', label='person', confidence=.9, bbox=[0,0,.4,1]),\n                   dict(index=1, instance_id='o1', label='horse', confidence=.8, bbox=[.4,0,1,1])]\n        relation = dict(subject_index=0, object_index=1, subject_instance='p0', object_instance='o1',\n                        score=.7, threshold=.5, text='person riding horse')\n        entry = dict(objects=objects, relations=[relation], mode='object_relation')\n        visual = np.ones((197, 768), dtype=np.float32)\n        f, roles, valid, texts = region_slots(entry, visual)\n        self.assertEqual(valid.sum(), 3)\n        self.assertEqual(texts[:3], ['person','horse','person riding horse'])\n        bounds=evidence_bounds(entry)\n        np.testing.assert_array_equal(bounds[0,2],bounds[0,0])\n        np.testing.assert_array_equal(bounds[1,2],bounds[1,1])\n        np.testing.assert_array_equal(bounds[2,2],[0,0,1,1])\n        self.assertEqual(phrase_identity('Person'),phrase_identity(' person '))\n        self.assertNotEqual(phrase_identity('horse'),phrase_identity('person'))\n        np.testing.assert_array_equal(roles[:3], [0,1,2])\n        np.testing.assert_array_equal(f[2,:2304], 1)\n        entry['relations'] = []; entry['mode'] = 'object_only'\n        self.assertEqual(region_slots(entry, visual)[2].sum(), 2)\n        entry['objects'] = []; entry['mode'] = 'no_objects'\n        self.assertFalse(region_slots(entry, visual)[2].any())\n        entry.update(objects=objects, relations=[dict(relation, subject_instance='wrong')], mode='object_relation')\n        with self.assertRaises(ValueError): region_slots(entry, visual)\n\n    def test_locked_split_is_deterministic_and_excludes_test(self):\n        data = {'images': [dict(filename=f'COCO_train2014_{i:012d}.jpg', split=split)\n                           for i, split in enumerate(['train']*6+['restval']*2+['val']*5+['test']*3)]}\n        a = cohorts(data, 4, 3); b = cohorts(data, 4, 3)\n        self.assertEqual(identity(a), identity(b))\n        self.assertTrue(all(x['image_id'] < 8 for x in a['train']))\n        self.assertTrue(all(8 <= x['image_id'] < 13 for x in a['val']))\n        self.assertFalse({r['coco_id'] for r in a['train']} & {r['coco_id'] for r in a['val']})\n\n\nif __name__ == '__main__': unittest.main(verbosity=2)\n", 'test_pilot_model.py': '"""Kaggle CPU preflight; no pretrained downloads or notebook launch."""\nimport unittest\nimport torch\nfrom torch import nn\nfrom pilot_model import EvidenceMemory, PilotModel\n\n\nclass FakeDecoder(nn.Module):\n    def __init__(self):\n        super().__init__();self.embedding=nn.Embedding(9,512);self.out=nn.Linear(512,9)\n    def forward(self,tgt,memory,tgt_key_padding_mask=None,memory_key_padding_mask=None):\n        keep=(~memory_key_padding_mask)[...,None]\n        value=(memory*keep).sum(1)/keep.sum(1).clamp_min(1)\n        return self.out(self.embedding(tgt)+value[:,None])\n\n\nclass FakeBase(nn.Module):\n    def __init__(self):\n        super().__init__();self.projection=nn.Linear(768,512);self.decoder=FakeDecoder()\n    def build_memory(self,visual,legacy,mask):\n        memory=torch.cat((legacy,self.projection(visual[:,:2])),1)\n        padding=torch.cat((mask.eq(0),torch.zeros(len(visual),2,dtype=torch.bool,device=visual.device)),1)\n        return memory,padding\n\n\ndef batch(device=\'cpu\'):\n    b=2\n    bounds=torch.tensor([0.,0.,1.,1.],device=device).expand(b,9,3,4).clone()\n    d=dict(visual=torch.randn(b,197,768,device=device),legacy=torch.randn(b,2,512,device=device),\n           legacy_mask=torch.ones(b,2,dtype=torch.long,device=device),text=torch.randn(b,9,512,device=device),\n           roles=torch.tensor([[0,1,2]*3]*b,device=device),valid=torch.ones(b,9,dtype=torch.bool,device=device),bounds=bounds,\n           phrase_ids=torch.arange(1,19,device=device).reshape(b,9),image_id=torch.arange(b,device=device),\n           captions=torch.tensor([[[1,4,5,2,0]]*5]*b,device=device),\n           caption_mask=torch.tensor([[[False,False,False,False,True]]*5]*b,device=device))\n    d.update(swap_text=d[\'text\'].flip(0),swap_visual=d[\'visual\'].flip(0),swap_bounds=bounds.flip(0),swap_valid=d[\'valid\'].clone())\n    return d\n\n\nclass EvidenceChecks(unittest.TestCase):\n    def test_attention_mask_and_nearest_patch(self):\n        torch.manual_seed(42);m=EvidenceMemory();d=batch()\n        bounds=d[\'bounds\'][0,:2].clone();bounds[0,2]=torch.tensor([0.,0.,.5,.5]);bounds[1,2]=torch.tensor([0.,0.,.001,.001])\n        _,_,a=m.attend(d[\'visual\'][0,1:].expand(2,-1,-1),d[\'text\'][0,:2],d[\'roles\'][0,:2],bounds)\n        outside=(m.xy[:,0]>.5)|(m.xy[:,1]>.5)\n        self.assertEqual(int(torch.count_nonzero(a[0,outside])),0)\n        self.assertEqual(int(torch.count_nonzero(a[1])),1)\n        torch.testing.assert_close(a.sum(-1),torch.ones(2))\n\n    def test_text_changes_attention_but_cannot_leak_into_constant_image_values(self):\n        torch.manual_seed(42);m=EvidenceMemory();d=batch();p=d[\'visual\'][0,1:].expand(2,-1,-1)\n        e,s,a=m.attend(p,d[\'text\'][0,:2],torch.ones(2,dtype=torch.long),d[\'bounds\'][0,:2])\n        self.assertGreater(float((a[0]-a[1]).abs().sum().detach()),1e-5)\n        constant=p[:,:1].expand_as(p)\n        e,_,_=m.attend(constant,d[\'text\'][0,:2],torch.ones(2,dtype=torch.long),d[\'bounds\'][0,:2])\n        torch.testing.assert_close(e[0],e[1],atol=2e-5,rtol=2e-5)\n\n    def test_alignment_eligibility_and_gradients(self):\n        torch.manual_seed(42);m=PilotModel(FakeBase(),\'E\');d=batch()\n        loss,_,_=m(d,None,None,None,0.,0);loss.backward()\n        self.assertEqual(m.last_alignment[\'pairs\'],18)\n        for part in (m.extra.query,m.extra.key,m.extra.value):\n            self.assertGreater(sum(float(p.grad.abs().sum()) for p in part.parameters() if p.grad is not None),0)\n        self.assertTrue(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\n        d[\'phrase_ids\'].fill_(1);m.zero_grad();loss,_,_=m(d,None,None,None,0.,0)\n        self.assertEqual(m.last_alignment[\'pairs\'],0)\n        self.assertTrue(torch.isfinite(loss));loss.backward()\n        d[\'valid\'].zero_();loss,_,_=m(d,None,None,None,0.,0)\n        self.assertEqual(m.last_alignment[\'pairs\'],0);self.assertTrue(torch.isfinite(loss))\n\n    def test_off_preserves_base_and_ce_scst_backward(self):\n        for arm in (\'A\',\'E\'):\n            torch.manual_seed(42);d=batch();m=PilotModel(FakeBase(),arm).eval()\n            old,pad=m.base.build_memory(d[\'visual\'],d[\'legacy\'],d[\'legacy_mask\'])\n            mem,mask=m.memory(d,\'off\');t=d[\'captions\'][:,0,:-1]\n            torch.testing.assert_close(m.decoder(t,old,memory_key_padding_mask=pad),m.decoder(t,mem,memory_key_padding_mask=mask))\n            loss,_,_=m(d,d[\'captions\'][:,0,:4],torch.ones(2,3,dtype=torch.bool),torch.tensor([1.,-.3]),.05,0)\n            loss.backward();self.assertTrue(torch.isfinite(loss))\n\n\nif __name__==\'__main__\': unittest.main(verbosity=2)\n', 'test_pilot_ddp.py': '"""Exactly two GPUs, one arm at a time. Not caption training."""\nimport os\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\nfrom pilot_model import PilotModel\nfrom test_pilot_model import FakeBase,batch\n\n\ndef main():\n    rank=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(rank);dist.init_process_group(\'nccl\')\n    try:\n        for arm in (\'A\',\'E\'):\n            torch.manual_seed(42);model=PilotModel(FakeBase(),arm).cuda().eval()\n            ddp=DistributedDataParallel(model,device_ids=[rank],find_unused_parameters=True,broadcast_buffers=False)\n            optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)\n            for i,weight in enumerate((0.,.05,0.)):\n                torch.manual_seed(90+rank);data=batch(f\'cuda:{rank}\')\n                if i==2 and rank==1: data[\'valid\'].zero_()\n                optimizer.zero_grad(set_to_none=True)\n                loss,_,_=ddp(data,data[\'captions\'][:,0,:4],torch.ones(2,3,dtype=torch.bool,device=rank),torch.tensor([1.,-.3],device=rank),weight,0)\n                loss.backward();optimizer.step()\n                vector=torch.cat([p.detach().flatten() for p in model.parameters()]);other=vector.clone();dist.broadcast(other,0)\n                torch.testing.assert_close(vector,other,rtol=0,atol=0)\n            if rank==0: print(f\'Two-GPU synchronized CE/SCST/alignment smoke PASS: {arm}\',flush=True)\n            del optimizer,ddp,model\n    finally: dist.destroy_process_group()\n\n\nif __name__==\'__main__\': main()\n', 'pilot_metric_smoke.py': '"""Fail on metric/tokenizer dependencies before staging or training."""\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport numpy as np\nfrom captioning.metrics import score_files\nfrom captioning.scst import TrainingCider\nfrom captioning.object_reward import _AnnotationReader\n\n\ndef main(chair):\n    references = {1: [\'a person holding an umbrella\']*5, 2: [\'a dog running through grass\']*5}\n    reward = TrainingCider(references, reference_cache_size=2).score([1,2],[r[0] for r in references.values()])\n    assert np.isfinite(reward).all() and (reward > 0).all()\n    with tempfile.TemporaryDirectory() as folder:\n        p = Path(folder)\n        gt = dict(info={},licenses=[],type=\'captions\',images=[{\'id\':i} for i in references],\n                  annotations=[dict(id=i*5+j,image_id=i,caption=text) for i,refs in references.items() for j,text in enumerate(refs)])\n        (p/\'gt.json\').write_text(json.dumps(gt))\n        (p/\'pred.json\').write_text(json.dumps([dict(image_id=i,caption=refs[0]) for i,refs in references.items()]))\n        result = score_files(p/\'pred.json\',p/\'gt.json\',p/\'metrics.json\')\n        assert np.isfinite(result[\'CIDEr\'])\n        with (Path(chair)/\'chair.pkl\').open(\'rb\') as stream: annotation = _AnnotationReader(stream).load()\n        ids = sorted(annotation.imid_to_objects)[:2]\n        (p/\'chair_input.json\').write_text(json.dumps([dict(image_id=i,caption=\'a person\') for i in ids]))\n        subprocess.run([sys.executable,\'-u\',str(Path(chair)/\'chair.py\'),\'--cache\',str(Path(chair)/\'chair.pkl\'),\n                        \'--cap_file\',str(p/\'chair_input.json\'),\'--save_path\',str(p/\'chair_output.json\')],cwd=chair,check=True)\n        assert \'overall_metrics\' in json.loads((p/\'chair_output.json\').read_text())\n    print(\'CIDEr reward, official caption metrics and CHAIR smoke PASS\',flush=True)\n\n\nif __name__ == \'__main__\': main(sys.argv[1])\n'}
SOURCE_HASHES = {'pilot_data.py': 'aac539b7c23f4acc3ac05b99496ef8eb79f740e6c99d11742b6de08fd2b892c0', 'pilot_model.py': '4f8189de7d183bb062247043d6a2ce7bcb6fbd9fb7d0d485922c065d8cbe962b', 'run_pilot.py': 'b8a58ecdccf89151aedbedad3ca0ad73832438db8bae707cdcf1aa5800830801', 'test_pilot_data.py': '5481c322e2266447028d93c694d765b3dbefb77428d6a9a9eea31d75422e1f8f', 'test_pilot_model.py': '49b59dcfa6caa41f9e1058856f65467bdc94635a2c6c8f45c89714d52b45353c', 'test_pilot_ddp.py': '757c2e77a5e57d3582e828fa8920547697da35eda7876ff49e3eeda9490d1943', 'pilot_metric_smoke.py': '1f33a26f3edcd21574189464633ac2060ac8cff0a69ec0af27505b07cecbadf6'}
for name, source in SOURCES.items(): (REPO/name).write_text(source,encoding='utf-8')
print('Embedded pilot source installed',flush=True)


## 2. Kiểm tra trước khi chạy
Kiểm tra pooling, mask, gradient CE/SCST, đồng bộ hai GPU và bộ tính điểm. Chưa huấn luyện caption tại bước này.

In [ ]:
for name in ('test_pilot_data.py','test_pilot_model.py'):
    run([sys.executable,'-u',REPO/name],cwd=REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
     '--max_restarts=0',REPO/'test_pilot_ddp.py'],cwd=REPO)
run([sys.executable,'-u',REPO/'pilot_metric_smoke.py',CHAIR],cwd=REPO)
print('PREFLIGHT PASS',flush=True)


## 3. Khóa tập ảnh và chuẩn bị đặc trưng
Chỉ stage 11.000 ảnh từ visual cache. Tạo embedding CLIP đóng băng cho cụm chủ thể/đối tượng/quan hệ; giữ liên kết instance và box. Bước này chủ yếu I/O, không cần cả hai GPU.

In [ ]:
CONFIG = Path('/tmp/semantic_evidence_pilot_config_' + ARM + '.json')
expected = {
    'checkpoint':'5f8eead3f4e960189345b1a9bf492763fe52ec71f70115b73244ffe9d03d2e75',
    'legacy_prompt':'d7910e9bf87e3aad86d806c77ec336b34793e3bef68a3f662fdcb2e9456ca38d',
    'upt_prompt':'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d',
    'dataset_json':'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d',
    'detections':'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c',
}
cfg = dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),upt_prompt=str(UPT_PROMPT),
           visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),detections=str(DETECTIONS),
           output=str(OUTPUT),stage='/tmp/semantic_evidence_stage_'+ARM+'.h5',chair=str(CHAIR),
           global_batch_size=8,eval_batch_size=16,warmup_updates=200,base_lr=5e-6,branch_lr=1e-4,
           scst_weight=.05,expected_hashes=expected,seed=42,project_commit=PROJECT_COMMIT,
           embedded_source_sha256=SOURCE_HASHES)
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u',REPO/'run_pilot.py','prepare','--config',CONFIG],cwd=REPO)
PROTOCOL = OUTPUT/'protocol.json'
(OUTPUT/'source').mkdir()
for name,source in SOURCES.items(): (OUTPUT/'source'/name).write_text(source,encoding='utf-8')
shutil.copy2(SETUP_LOG,OUTPUT/'setup.log')
locked = json.loads(PROTOCOL.read_text())
print('LOCKED COHORT:',locked['cohort_sha256'],locked['counts_by_mode'],flush=True)


## 4. Huấn luyện nhánh A bằng hai GPU
        Hai worker thuộc cùng nhánh, đồng bộ gradient DDP; GPU 0 không chạy nhánh khác GPU 1.
        Đánh giá checkpoint ban đầu trước, sau đó 1.250 cập nhật và đánh giá checkpoint cuối.
        Không chọn checkpoint bằng test; checkpoint cuối được lưu riêng ở Output.

In [ ]:
TRAIN_LOG = OUTPUT/'run.log'
try:
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
         '--max_restarts=0',REPO/'run_pilot.py','worker','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
except Exception:
    run([sys.executable,'-u',REPO/'run_pilot.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
    raise


## 5. Tính điểm và đóng gói kết quả
        CIDEr/BLEU/METEOR/ROUGE và CHAIR trên cùng 1.000 val, giải mã greedy cho mọi nhánh.
        E có thêm tắt nhánh, tráo text, tráo vùng; đây là kiểm tra mức sử dụng bằng chứng, chưa phải kiểm định quan hệ đúng/sai.
        Gửi lại ZIP review; giữ `checkpoint.pth` trong Saved Output. ZIP review không chứa checkpoint hay HDF5.

In [ ]:
try:
    run([sys.executable,'-u',REPO/'run_pilot.py','evaluate','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
finally:
    run([sys.executable,'-u',REPO/'run_pilot.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
result = json.loads((OUTPUT/'run_summary.json').read_text())
print(json.dumps(result,indent=2,ensure_ascii=False),flush=True)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.pth',flush=True)
